In [ ]:
# Removed: google.colab, cv2, Augmentor, pickle, Kaggle download
# Images already cleaned/cropped in Research/Data/galaxies_cropped/
import numpy as np
import pandas as pd
import scipy.ndimage
import random
import matplotlib.pyplot as plt
import skimage as sk
import skimage.transform
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

RANDOM = 42
random.seed(RANDOM)
np.random.seed(RANDOM)
torch.manual_seed(RANDOM)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

DATA = '../Research/Data/'
CROP = DATA + 'galaxies_cropped/'
IMG_SIZE = 64  # resize target on load; original pre-processed images to 64x64

In [ ]:
# Removed: original's label-loading and GalaxyID-.jpg renaming; our files are already .png
labels = pd.read_csv(DATA + 'galaxies_training/training_solutions_rev1.csv')
classes = labels.columns[1:]  # 37 morphology class columns

# Map integer GalaxyID to filenames in galaxies_cropped/
filenames = labels.GalaxyID.apply(lambda x: str(int(x)) + '.png').to_numpy()
targets = labels[classes].to_numpy(dtype=np.float32)

X_train, X_test, y_train, y_test = train_test_split(
    filenames, targets, test_size=0.20, random_state=RANDOM
)
print(f'Training: {len(X_train)}, Test: {len(X_test)}')

def RMSE(pred, truth):
    return np.sqrt(np.mean(np.square(np.array(pred).flatten() - np.array(truth).flatten())))

In [ ]:
def get_output_width(width, kernel, padding, stride):
    return int((width + 2 * padding - kernel - 1) / stride + 1)


def torch_batches(pics, labels, path=CROP, batch_size=30, rotate=False):
    """
    Generates (image_tensor, label_tensor, filenames) batches.

    Changed from original:
    - Loads uint16 grayscale PNGs via skimage (was plt.imread RGB JPEGs)
    - Normalizes uint16 [0, 65535] -> float32 [0, 1]
    - Resizes to IMG_SIZE x IMG_SIZE on load (was pre-processed offline to 64x64)
    - Output shape: (N, 1, H, W) via unsqueeze(1) instead of (N, 3, H, W) via permute
    - Rotation still uses scipy.ndimage.rotate on the 2D grayscale array
    """
    labels_t = torch.tensor(labels, dtype=torch.float32)
    l = len(pics)
    batches = int(l / batch_size)
    leftover = l % batch_size

    def load_batch(batch_pics):
        imgs = []
        for pic in batch_pics:
            img = sk.io.imread(path + pic).astype(np.float32) / 65535.0
            img = sk.transform.resize(
                img, (IMG_SIZE, IMG_SIZE), preserve_range=True, anti_aliasing=True
            ).astype(np.float32)
            if rotate:
                img = scipy.ndimage.rotate(
                    img, np.random.randint(0, 360), reshape=False
                ).astype(np.float32)
            imgs.append(img)
        # unsqueeze(1) adds channel dim: (N, H, W) -> (N, 1, H, W)
        return torch.tensor(np.array(imgs), dtype=torch.float32).unsqueeze(1)

    for batch in range(batches):
        s = batch * batch_size
        yield load_batch(pics[s:s + batch_size]), labels_t[s:s + batch_size], pics[s:s + batch_size]

    if leftover > 0:
        s = batches * batch_size
        yield load_batch(pics[s:]), labels_t[s:], pics[s:]

In [ ]:
# Architecture: Attempt 4 from original — achieved RMSE 0.11025 on 64x64 color
# Changed: c1_in=1 (grayscale single channel) instead of 3 (RGB)
# All other hyperparameters kept identical.
pool_kernel = 2
stride = 1

c1_in = 1       # changed: grayscale (original was 3 for color)
c1_kernel = 9
c1_out = 64
c2_kernel = 5
c2_out = 96
c3_kernel = 3
c3_out = 128

# Float-division padding in get_output_width compensates for the formula offset
# so calculated sizes match actual PyTorch output sizes.
c1_pooled_width = get_output_width(
    get_output_width(IMG_SIZE, c1_kernel, c1_kernel / 2, stride),
    pool_kernel / 2, 0, pool_kernel)
c2_pooled_width = get_output_width(
    get_output_width(c1_pooled_width, c2_kernel, c2_kernel / 2, stride),
    pool_kernel, pool_kernel / 2, pool_kernel)
c3_pooled_width = get_output_width(
    get_output_width(c2_pooled_width, c3_kernel, c3_kernel / 2, stride),
    pool_kernel, pool_kernel / 2, pool_kernel)

full_1_in = c3_out * c3_pooled_width * c3_pooled_width
full_1_out, full_2_out = 1024, 256
full_3_out = len(classes)

print(f'P1={c1_pooled_width}, P2={c2_pooled_width}, P3={c3_pooled_width}, FC1_in={full_1_in}')


class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(c1_in,  c1_out, c1_kernel, stride, int(c1_kernel / 2), padding_mode='zeros')
        self.conv2 = nn.Conv2d(c1_out, c2_out, c2_kernel, stride, int(c2_kernel / 2), padding_mode='zeros')
        self.conv3 = nn.Conv2d(c2_out, c3_out, c3_kernel, stride, int(c3_kernel / 2), padding_mode='zeros')
        self.pool     = nn.MaxPool2d(pool_kernel)
        self.conv2_bn = nn.BatchNorm2d(c2_out)
        self.conv3_bn = nn.BatchNorm2d(c3_out)
        self.fc1      = nn.Linear(full_1_in, full_1_out)
        self.fc2      = nn.Linear(full_1_out, full_2_out)
        self.fc3      = nn.Linear(full_2_out, full_3_out)
        self.dropout  = nn.Dropout(p=0.15)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2_bn(self.conv2(x))))
        x = self.pool(F.relu(self.conv3_bn(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.dropout(self.fc1(x)))
        x = F.relu(self.dropout(self.fc2(x)))
        return torch.sigmoid(self.fc3(x))  # changed: F.sigmoid -> torch.sigmoid (F.sigmoid deprecated)


net = Net().to(device)
print(net)

In [ ]:
# Training
# Removed: save/load to Google Drive, Colab display helpers (clear_output, display)
# Changed: tensors moved to `device`; print uses \r for in-place progress
loss_weight = torch.tensor(
    np.sum(y_train, axis=0) / np.sum(y_train), dtype=torch.float32
).to(device)


def weighted_mse_loss(output, targets, weights):
    loss = (output - targets) ** 2
    loss = loss * weights.expand(loss.shape)
    return loss.mean(0).sum()


optimizer = optim.Adam(net.parameters())

loss_history = []
batch_size = 1024
mini_batch_size = 64

net.train()
for epoch in range(1, 6):
    batch_no = 1
    datagen = torch_batches(X_train, y_train, batch_size=batch_size, rotate=True)
    for images, batch_targets, _ in datagen:
        images, batch_targets = images.to(device), batch_targets.to(device)
        for i in range(int(batch_size / mini_batch_size)):
            s = i * mini_batch_size
            if s >= images.shape[0]:
                break
            optimizer.zero_grad()
            outputs = net(images[s:s + mini_batch_size])
            loss = weighted_mse_loss(outputs, batch_targets[s:s + mini_batch_size], loss_weight)
            loss.backward()
            optimizer.step()
        print(f'Epoch {epoch}, batch {batch_no} — loss: {loss.item():.5f}', end='\r')
        batch_no += 1
        loss_history.append((epoch, batch_no, loss.item()))
    print()

torch.save(net.state_dict(), 'galaxy_zoo_cnn.pt')

In [ ]:
# Evaluation
net.eval()
pred = []
with torch.no_grad():
    for images, _, _ in torch_batches(X_test, y_test, batch_size=1000):
        pred.append(net(images.to(device)).cpu().numpy())

pred = np.concatenate(pred, axis=0)
print(f'Test RMSE: {RMSE(pred, y_test):.5f}')

In [ ]:
# Loss curve
plt.figure(figsize=(10, 4))
plt.title('Training Loss (Weighted MSE)')
plt.xlabel('Batch')
plt.ylabel('Loss')
plt.plot([h[2] for h in loss_history[3:]])  # skip first 3 noisy batches
plt.tight_layout()
plt.show()